# m39 lab · Learning on maps and electrograms

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You build graph-convolution layers on a mapping mesh, train a small graph network to find slow regions on
synthetic maps, handle rare targets, and then test a classifier on real intracardiac recordings with splits that
respect patients.

| Part | What you do | Data |
| --- | --- | --- |
| A | Build $\hat A$ on a mesh; check eq-m39-gcn and eq-m39-smoothing; smooth a noisy map | synthetic |
| B | Synthetic mapping cases with slow regions; per-point features | synthetic |
| C | A per-point model against a NumPy graph network: ROC AUC, PR AUC, Dice, centroid distance | synthetic |
| D | Rare targets: cross-entropy, weighted cross-entropy and focal loss | synthetic |
| E | Catheter position from electrogram features: window-wise against leave-one-patient-out | PhysioNet |
| F | Optional: the graph network in PyTorch | synthetic |

The synthetic maps use the eikonal model of m30 with conduction speeds of 1.0 m/s in normal tissue and 0.6 m/s in
the slow region (values reported for normal-voltage and low-voltage left atrium in persistent AF, Heil et al.
2023). Region sizes, timing noise and the voltage-like feature are illustrative. Each part ends with **checks**.
The lab runs in about a minute, plus downloads.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

from epml import graphs as gr
from epml import mapping as mp
from epml import ml
from epml.data import DataUnavailable

## Part A · Graph layers on a mesh

A 21-by-21 grid of points 2 mm apart, triangulated as in m30. Build the adjacency, the normalized
$\hat A = \tilde D^{-1/2}(A+I)\tilde D^{-1/2}$ (eq-m39-gcn), and check that $\sqrt{\tilde d}$ is its fixed point
(eq-m39-smoothing). Then smooth a noisy map with 0, 2 and 8 layers.

In [ ]:
NX, H = 21, 2.0
verts, tris = mp.plane_mesh(NX, NX, H)
A = gr.mesh_adjacency(tris, len(verts))
Ah = gr.normalized_adjacency(A)
areas = gr.vertex_areas(verts, tris)
v = gr.stationary_vector(A)
ev = np.linalg.eigvalsh(Ah.toarray())
print(f"{len(verts)} vertices, degrees {np.unique(np.asarray(A.sum(1)).ravel())}; eigenvalues of A_hat in "
      f"[{ev.min():.3f}, {ev.max():.3f}]")

rng = np.random.default_rng(39)
xy = verts[:, :2]
truth = np.where(np.hypot(*(xy - 20).T) <= 7, 0.2, 1.0)        # a disc of low values (illustrative units)
noisy = truth + 0.3 * rng.normal(size=truth.size)
fig, axs = plt.subplots(1, 3, figsize=(9, 3))
for ax, L in zip(axs, [0, 2, 8]):
    f = noisy.copy()
    for _ in range(L):
        f = Ah @ f
    ax.tripcolor(xy[:, 0], xy[:, 1], tris, f, vmin=0, vmax=1.2, shading="gouraud")
    ax.set(title=f"{L} layers, RMS error {np.sqrt(np.mean((f - truth) ** 2)):.2f}", aspect="equal", xticks=[], yticks=[])
plt.show()

In [ ]:
# Checks
assert np.allclose(Ah @ v, v)
assert ev.max() < 1 + 1e-9 and ev.min() > -1
smooth2 = Ah @ (Ah @ noisy)
assert np.sqrt(np.mean((smooth2 - truth) ** 2)) < np.sqrt(np.mean((noisy - truth) ** 2))   # 2 layers remove noise

## Part B · Synthetic mapping cases

Each case: a pacing site at one corner, a slow disc of random size and place, activation times from the eikonal
model with 1 ms of timing noise, and two features per point:

1. a voltage-like value, lower inside the slow region, with patient-to-point noise (illustrative);
2. a local speed estimate from a plane fit of the activation times over the point and its mesh neighbours.

The label is "inside the slow region". Forty cases: 28 to train, 12 to test (a split by case, as by patient).

In [ ]:
neigh = [A.indices[A.indptr[i]:A.indptr[i + 1]] for i in range(len(verts))]
cent = verts[tris].mean(axis=1)[:, :2]

def make_case(rng, rmin=4.0, rmax=8.0):
    c = rng.uniform(8, 32, 2)
    R = rng.uniform(rmin, rmax)
    slow_tri = np.hypot(*(cent - c).T) <= R
    T = mp.eikonal(verts, tris, [0], np.where(slow_tri, 0.6, 1.0))      # ms; 1 m/s = 1 mm/ms
    y = (np.hypot(*(xy - c).T) <= R).astype(float)
    Tm = T + rng.normal(0, 1.0, T.size)
    volt = np.exp(rng.normal(0, 0.35, T.size)) * np.where(y == 1, 0.6, 1.0)
    speed = np.empty(T.size)
    for i in range(T.size):
        nb = np.r_[i, neigh[i]]
        g = np.linalg.lstsq(np.c_[xy[nb], np.ones(nb.size)], Tm[nb], rcond=None)[0][:2]
        speed[i] = 1.0 / max(np.linalg.norm(g), 1e-3)
    X = np.c_[np.log(volt), np.log(np.clip(speed, 0.05, 5))]
    return X, y, c, R

rng = np.random.default_rng(390)
cases = [make_case(rng) for _ in range(40)]
mu = np.concatenate([c[0] for c in cases[:28]]).mean(0)
sd = np.concatenate([c[0] for c in cases[:28]]).std(0)
cases = [((X - mu) / sd, y, c, R) for X, y, c, R in cases]
train, test = cases[:28], cases[28:]
print(f"targets cover {np.mean([y.mean() for _, y, _, _ in cases]):.1%} of points on average")

X0, y0, c0, R0 = test[0]
fig, axs = plt.subplots(1, 2, figsize=(7, 3))
for ax, f, title in zip(axs, X0.T, ["log voltage-like value (z)", "log local speed (z)"]):
    ax.tripcolor(xy[:, 0], xy[:, 1], tris, f, shading="gouraud")
    ax.add_patch(plt.Circle(c0, R0, fill=False, color="w", lw=1.5))
    ax.set(title=title, aspect="equal", xticks=[], yticks=[])
plt.show()

In [ ]:
# Checks
assert len(train) == 28 and len(test) == 12
assert 0.02 < np.mean([y.mean() for _, y, _, _ in cases]) < 0.15

## Part C · One point at a time, or the whole map

Model 1: logistic regression on each point's two features. Model 2: a two-layer graph network in NumPy
(eq-m39-gcn), trained with the cross-entropy and the Adam update. Judge both on the 12 test cases by ROC AUC, PR AUC
(average precision), Dice at a 0.5 threshold (eq-m39-dice, area-weighted) and the distance between predicted and
true centroids.

In [ ]:
def evaluate(prob_fn, cases):
    out = {"ROC AUC": [], "PR AUC": [], "Dice": [], "centroid distance (mm)": []}
    for X, y, c, R in cases:
        p = prob_fn(X)
        out["ROC AUC"].append(ml.auc_rank(p, y))
        out["PR AUC"].append(average_precision_score(y, p))
        pred = p > 0.5
        out["Dice"].append(gr.dice(pred, y == 1, areas))
        if pred.any():
            cp = (xy[pred] * areas[pred, None]).sum(0) / areas[pred].sum()
            out["centroid distance (mm)"].append(np.hypot(*(cp - c)))
    return {k: float(np.median(v)) if "distance" in k else float(np.mean(v)) for k, v in out.items()}

logit = LogisticRegression(max_iter=1000).fit(np.concatenate([X for X, _, _, _ in train]),
                                              np.concatenate([y for _, y, _, _ in train]))
gcn = gr.GCN(2, 8, rng=np.random.default_rng(1))
history = gcn.fit([(Ah, X, y) for X, y, _, _ in train], epochs=300, lr=0.02)
results = {"per-point logistic": evaluate(lambda X: logit.predict_proba(X)[:, 1], test),
           "graph network": evaluate(lambda X: gcn.predict_proba(Ah, X), test)}
for name, r in results.items():
    print(f"{name:20s} " + "  ".join(f"{k} {v:.3f}" for k, v in r.items()))

In [ ]:
# Checks
g, l = results["graph network"], results["per-point logistic"]
assert history[-1] < history[0]
assert g["ROC AUC"] > l["ROC AUC"] and g["PR AUC"] > l["PR AUC"] + 0.1
assert g["Dice"] > l["Dice"] + 0.1

## Part D · Rare targets

Shrink the slow regions to 2–3 mm, so targets cover about 1% of points. Train the same graph network three ways:
plain cross-entropy, cross-entropy with positives weighted 20 times, and the focal loss with $\alpha = 0.75$,
$\gamma = 2$ (eq-m39-focal; the values used by Guler et al. 2026). Compare ROC AUC with PR AUC, and recall and
precision at a 0.5 threshold.

In [ ]:
rng = np.random.default_rng(391)
rare = [make_case(rng, 2.0, 3.0) for _ in range(40)]
rare = [((X - mu) / sd, y, c, R) for X, y, c, R in rare]
r_train, r_test = rare[:28], rare[28:]
print(f"targets cover {np.mean([y.mean() for _, y, _, _ in rare]):.2%} of points")
rare_results = {}
for name, kw in [("cross-entropy", {}), ("weighted x20", {"loss": "bce", "weight": 20.0}),
                 ("focal (0.75, 2)", {"loss": "focal", "alpha": 0.75, "gamma": 2.0})]:
    net = gr.GCN(2, 8, rng=np.random.default_rng(1))
    net.fit([(Ah, X, y) for X, y, _, _ in r_train], epochs=300, lr=0.02, **kw)
    P = np.concatenate([net.predict_proba(Ah, X) for X, _, _, _ in r_test])
    Y = np.concatenate([y for _, y, _, _ in r_test])
    flag = P > 0.5
    rare_results[name] = {"ROC AUC": ml.auc_rank(P, Y), "PR AUC": average_precision_score(Y, P),
                          "recall@0.5": float(flag[Y == 1].mean()),
                          "precision@0.5": float(Y[flag].mean()) if flag.any() else float("nan")}
for name, r in rare_results.items():
    print(f"{name:16s} " + "  ".join(f"{k} {v:.3f}" for k, v in r.items()))

In [ ]:
# Checks
ce, wt, fo = (rare_results[k] for k in ["cross-entropy", "weighted x20", "focal (0.75, 2)"])
assert ce["recall@0.5"] < 0.1                          # plain cross-entropy almost never says "target"
assert wt["recall@0.5"] > 0.3 and fo["recall@0.5"] > 0.2
assert all(r["PR AUC"] < r["ROC AUC"] - 0.3 for r in rare_results.values())   # ROC AUC flatters rare targets

## Part E · Real recordings, honest splits

The Intracardiac Atrial Fibrillation Database (PhysioNet, doi:10.13026/C23S33, Open Data Commons Attribution
v1.0) has 32 records from 8 patients in AF, with the catheter in four positions per patient; record names end in
`svc`, `ivc`, `tva` and `afw`. Amplitudes are uncalibrated, so the features below ignore amplitude.

Each record holds four surface ECG leads or intracardiac bipoles labeled CS12 (distal) to CS90 (proximal) on a
decapolar catheter; the label names the bipole, not the coronary sinus. The free-wall records are much longer and
include drug delivery and washout, so we use only the first 56 s of every record: fourteen 4-s windows each.

Question: can three electrogram features tell the catheter position? For each window, from the distal bipole CS12: the dominant frequency and the organization index of the rectified
signal (m29), and a shape measure (RMS of the derivative over RMS of the signal). The filter and search settings are
illustrative analysis choices; m29 notes that published values still need a source.

Two ways to test a classifier: a random split of windows (5-fold) and leave-one-patient-out. Without internet
access this part is skipped.

In [ ]:
from scipy.signal import decimate

from epml import complexmaps as cm
from epml.data import fetch_iafdb

SITES = ["svc", "ivc", "tva", "afw"]
WIN = 4.0                                     # s, illustrative
N_WIN = 14                                    # first 56 s of each record
rows = []
try:
    import wfdb
    for patient in range(1, 9):
        for site in SITES:
            rec = wfdb.rdrecord(fetch_iafdb(f"iaf{patient}_{site}"))
            names = [n.strip().upper() for n in rec.sig_name]
            if "CS12" not in names:
                raise ValueError(f"iaf{patient}_{site}: no CS12 channel among {rec.sig_name}")
            x, fs = rec.p_signal[:, names.index("CS12")], rec.fs
            starts = range(0, int(len(x) - WIN * fs) + 1, int(WIN * fs))
            for s in list(starts)[:N_WIN]:
                w = x[s: s + int(WIN * fs)]
                env = cm.preprocess_rectified(w, fs, band=(40, 250), lowpass=20)
                env = decimate(env, 10)                  # 1 kHz -> 100 Hz for the spectrum (illustrative)
                df = cm.dominant_frequency(env, fs / 10, 3, 12)
                oi = cm.organization_index(env, fs / 10, 3, 12, halfwidth=0.75)
                shape = np.sqrt(np.mean(np.diff(w) ** 2)) / (np.sqrt(np.mean(w ** 2)) + 1e-12)
                rows.append((patient, SITES.index(site), df, oi, shape))
    REAL = True
except DataUnavailable as err:
    print("Part E skipped:", err)
    REAL = False

if REAL:
    data = np.array(rows)
    pid, site, F = data[:, 0].astype(int), data[:, 1].astype(int), data[:, 2:]
    print(f"{len(data)} windows from {len(np.unique(pid))} patients")

In [ ]:
if REAL:
    from sklearn.model_selection import KFold, LeaveOneGroupOut, cross_val_predict
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    pred_win = cross_val_predict(model, F, site, cv=KFold(5, shuffle=True, random_state=0))
    pred_pat = cross_val_predict(model, F, site, cv=LeaveOneGroupOut(), groups=pid)
    acc_win = float(np.mean(pred_win == site))
    acc_pat = float(np.mean(pred_pat == site))
    print(f"accuracy, 4 positions (chance about 0.25): window-wise split {acc_win:.2f}, leave-one-patient-out {acc_pat:.2f}")
    for k, s in enumerate(SITES):
        print(f"  {s}: median dominant frequency {np.median(F[site == k, 0]):.2f} Hz over {np.sum(site == k)} windows")

In [ ]:
# Checks
if REAL:
    assert len(np.unique(pid)) == 8 and set(np.unique(site)) == {0, 1, 2, 3} and len(data) > 300
    assert acc_win >= acc_pat            # the window-wise split never looks worse here

## Part F · Optional: the graph network in PyTorch

The same two-layer network with PyTorch's automatic differentiation and a sparse $\hat A$. Runs only if PyTorch is
installed (`pip install torch`).

In [ ]:
try:
    import torch
except ImportError:
    torch = None
    if os.environ.get("EPML_REQUIRE_TORCH") == "1":
        raise
    print("PyTorch is not installed: part F skipped.")

if torch is not None:
    torch.manual_seed(0)
    coo = Ah.tocoo()
    At = torch.sparse_coo_tensor(np.vstack([coo.row, coo.col]), coo.data, coo.shape, dtype=torch.float32)
    W1 = torch.nn.Linear(2, 8)
    W2 = torch.nn.Linear(8, 1)
    params = list(W1.parameters()) + list(W2.parameters())
    opt = torch.optim.Adam(params, lr=0.02)
    lossf = torch.nn.BCEWithLogitsLoss()
    tensors = [(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)) for X, y, _, _ in train]
    def forward(Xt):
        h = torch.relu(W1(torch.sparse.mm(At, Xt)))      # eq-m39-gcn, layer 1
        return W2(torch.sparse.mm(At, h)).squeeze(1)      # layer 2
    for epoch in range(300):
        opt.zero_grad()
        loss = sum(lossf(forward(Xt), yt) for Xt, yt in tensors) / len(tensors)
        loss.backward()
        opt.step()
    with torch.no_grad():
        torch_res = evaluate(lambda X: torch.sigmoid(forward(torch.tensor(X, dtype=torch.float32))).numpy(), test)
    print("PyTorch graph network " + "  ".join(f"{k} {v:.3f}" for k, v in torch_res.items()))

In [ ]:
# Checks
if torch is not None:
    assert torch_res["ROC AUC"] > 0.95

## Where to go next

- In Part C, give the graph network 1 layer, then 4. Which changes Dice more: too few layers or too many?
- In Part D, try $\gamma = 0$ in the focal loss. Which result does it reproduce?
- In Part E, which feature separates positions best within one patient? Does that hold across patients?